Environment (Once)

In [ ]:
!pip install -q torch transformers accelerate


PHASE 1 — gpt2 (Baseline Check)

Goal: confirm KV cache + attention access.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_attentions=True,
    use_cache=True
).to(device)

model.eval()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
# Tokenize input
inputs = tokenizer("Hello world", return_tensors="pt").to(device)

# Forward pass with attentions
with torch.no_grad():
    outputs = model(**inputs)

attentions = outputs.attentions

print("Number of layers:", len(attentions))

# Inspect layer 0
attn0 = attentions[0]

print("Layer 0 attention shape:", attn0.shape)

# Break down dimensions
B, H, Q, K = attn0.shape
print("Batch size:", B)
print("Number of heads:", H)
print("Query length:", Q)
print("Key length:", K)


Number of layers: 12
Layer 0 attention shape: torch.Size([1, 12, 2, 2])
Batch size: 1
Number of heads: 12
Query length: 2
Key length: 2


**Sanity Test**

**Full Cache Expert**

In [ ]:
# -------------------------------------------------
# Full Cache Expert
# -------------------------------------------------
class FullCacheExpert:
    def update(self, past_layer, new_layer, layer_idx):
        return new_layer  # keep full cache unchanged


expert = FullCacheExpert()


# -------------------------------------------------
# Autoregressive Decoding Loop
# -------------------------------------------------
inputs = tokenizer("Hello, I am Ahad. Who are you?", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
past_kv = None

for step in range(3):

    if past_kv is None:
        position_ids = torch.zeros((1, 1), device=device, dtype=torch.long)
    else:
        position_ids = torch.tensor(
            [[past_kv.layers[0].keys.shape[2]]],
            device=device
        )

    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            position_ids=position_ids,
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_cache = out.past_key_values

    # Apply FullCacheExpert layer-by-layer
    for i, layer in enumerate(new_cache.layers):
        updated_layer = expert.update(
            past_kv.layers[i] if past_kv else None,
            layer,
            i
        )
        new_cache.layers[i] = updated_layer

    past_kv = new_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(f"\nStep {step}")
    for i, layer in enumerate(past_kv.layers):
        print(f"  Layer {i}: seq_len = {layer.keys.shape[2]}")


Step 0
  Layer 0: seq_len = 1
  Layer 1: seq_len = 1
  Layer 2: seq_len = 1
  Layer 3: seq_len = 1
  Layer 4: seq_len = 1
  Layer 5: seq_len = 1
  Layer 6: seq_len = 1
  Layer 7: seq_len = 1
  Layer 8: seq_len = 1
  Layer 9: seq_len = 1
  Layer 10: seq_len = 1
  Layer 11: seq_len = 1

Step 1
  Layer 0: seq_len = 2
  Layer 1: seq_len = 2
  Layer 2: seq_len = 2
  Layer 3: seq_len = 2
  Layer 4: seq_len = 2
  Layer 5: seq_len = 2
  Layer 6: seq_len = 2
  Layer 7: seq_len = 2
  Layer 8: seq_len = 2
  Layer 9: seq_len = 2
  Layer 10: seq_len = 2
  Layer 11: seq_len = 2

Step 2
  Layer 0: seq_len = 3
  Layer 1: seq_len = 3
  Layer 2: seq_len = 3
  Layer 3: seq_len = 3
  Layer 4: seq_len = 3
  Layer 5: seq_len = 3
  Layer 6: seq_len = 3
  Layer 7: seq_len = 3
  Layer 8: seq_len = 3
  Layer 9: seq_len = 3
  Layer 10: seq_len = 3
  Layer 11: seq_len = 3


**Layer-Sparse Expert**

In [ ]:
class LayerSparseExpert:
    def update(self, past_layer, new_layer, layer_idx):
        # zero out entire layer (keep shape)
        new_layer.keys = torch.zeros_like(new_layer.keys)
        new_layer.values = torch.zeros_like(new_layer.values)
        return new_layer


inputs = tokenizer("Hello", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
past_kv = None

expert = LayerSparseExpert()

for step in range(3):

    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_cache = out.past_key_values

    # Apply expert layer-by-layer
    for i, layer in enumerate(new_cache.layers):
        updated = expert.update(
            past_kv.layers[i] if past_kv else None,
            layer,
            i
        )
        new_cache.layers[i] = updated

    past_kv = new_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(f"\nStep {step}")
    for i, layer in enumerate(past_kv.layers):
        print(f"  Layer {i}: seq_len = {layer.keys.shape[2]}")



Step 0
  Layer 0: seq_len = 1
  Layer 1: seq_len = 1
  Layer 2: seq_len = 1
  Layer 3: seq_len = 1
  Layer 4: seq_len = 1
  Layer 5: seq_len = 1
  Layer 6: seq_len = 1
  Layer 7: seq_len = 1
  Layer 8: seq_len = 1
  Layer 9: seq_len = 1
  Layer 10: seq_len = 1
  Layer 11: seq_len = 1

Step 1
  Layer 0: seq_len = 2
  Layer 1: seq_len = 2
  Layer 2: seq_len = 2
  Layer 3: seq_len = 2
  Layer 4: seq_len = 2
  Layer 5: seq_len = 2
  Layer 6: seq_len = 2
  Layer 7: seq_len = 2
  Layer 8: seq_len = 2
  Layer 9: seq_len = 2
  Layer 10: seq_len = 2
  Layer 11: seq_len = 2

Step 2
  Layer 0: seq_len = 3
  Layer 1: seq_len = 3
  Layer 2: seq_len = 3
  Layer 3: seq_len = 3
  Layer 4: seq_len = 3
  Layer 5: seq_len = 3
  Layer 6: seq_len = 3
  Layer 7: seq_len = 3
  Layer 8: seq_len = 3
  Layer 9: seq_len = 3
  Layer 10: seq_len = 3
  Layer 11: seq_len = 3


**Head-Specific Expert**

In [ ]:
class HeadSpecificExpert:
    def __init__(self, keep_heads, window=1):
        self.keep_heads = keep_heads
        self.window = window

    def update(self, past_layer, new_layer, layer_idx):
        k = new_layer.keys
        v = new_layer.values

        B, H, S, D = k.shape

        new_k = []
        new_v = []

        for h in range(H):
            if h in self.keep_heads:
                kh = k[:, h:h+1, :, :]
                vh = v[:, h:h+1, :, :]
            else:
                kh = k[:, h:h+1, -self.window:, :]
                vh = v[:, h:h+1, -self.window:, :]

                pad_len = S - kh.shape[2]
                if pad_len > 0:
                    kh = torch.cat(
                        [torch.zeros(B, 1, pad_len, D, device=k.device), kh],
                        dim=2
                    )
                    vh = torch.cat(
                        [torch.zeros(B, 1, pad_len, D, device=v.device), vh],
                        dim=2
                    )

            new_k.append(kh)
            new_v.append(vh)

        new_layer.keys = torch.cat(new_k, dim=1)
        new_layer.values = torch.cat(new_v, dim=1)

        return new_layer
inputs = tokenizer("Hello", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
past_kv = None

expert = HeadSpecificExpert(keep_heads=[0], window=1)

for step in range(3):

    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_cache = out.past_key_values

    for i, layer in enumerate(new_cache.layers):
        if i == 1:  # apply only to layer 1
            updated = expert.update(
                past_kv.layers[i] if past_kv else None,
                layer,
                i
            )
            new_cache.layers[i] = updated

    past_kv = new_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(f"\nStep {step}")
    for i, layer in enumerate(past_kv.layers):
        print(f"  Layer {i}:")
        for h in range(layer.keys.shape[1]):
            print(f"    Head {h}: seq_len = {layer.keys[:, h].shape[1]}")



Step 0
  Layer 0:
    Head 0: seq_len = 1
    Head 1: seq_len = 1
    Head 2: seq_len = 1
    Head 3: seq_len = 1
    Head 4: seq_len = 1
    Head 5: seq_len = 1
    Head 6: seq_len = 1
    Head 7: seq_len = 1
    Head 8: seq_len = 1
    Head 9: seq_len = 1
    Head 10: seq_len = 1
    Head 11: seq_len = 1
  Layer 1:
    Head 0: seq_len = 1
    Head 1: seq_len = 1
    Head 2: seq_len = 1
    Head 3: seq_len = 1
    Head 4: seq_len = 1
    Head 5: seq_len = 1
    Head 6: seq_len = 1
    Head 7: seq_len = 1
    Head 8: seq_len = 1
    Head 9: seq_len = 1
    Head 10: seq_len = 1
    Head 11: seq_len = 1
  Layer 2:
    Head 0: seq_len = 1
    Head 1: seq_len = 1
    Head 2: seq_len = 1
    Head 3: seq_len = 1
    Head 4: seq_len = 1
    Head 5: seq_len = 1
    Head 6: seq_len = 1
    Head 7: seq_len = 1
    Head 8: seq_len = 1
    Head 9: seq_len = 1
    Head 10: seq_len = 1
    Head 11: seq_len = 1
  Layer 3:
    Head 0: seq_len = 1
    Head 1: seq_len = 1
    Head 2: seq_len = 1
    He

**Top-K Token Expert**

In [ ]:
class TopKTokenExpert:
    def __init__(self, K=16):
        self.K = K

    def update(self, past_layer, new_layer, layer_idx):
        k = new_layer.keys
        v = new_layer.values

        B, H, S, D = k.shape

        if S > self.K:
            mask = torch.zeros(S, device=k.device)
            mask[-self.K:] = 1.0
            mask = mask.view(1, 1, S, 1)

            new_layer.keys = k * mask
            new_layer.values = v * mask

        return new_layer


In [ ]:
inputs = tokenizer("Hello", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
past_kv = None

expert = TopKTokenExpert(K=2)

for step in range(3):

    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_cache = out.past_key_values

    for i, layer in enumerate(new_cache.layers):
        updated = expert.update(
            past_kv.layers[i] if past_kv else None,
            layer,
            i
        )
        new_cache.layers[i] = updated

    past_kv = new_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(f"\nStep {step}")
    for i, layer in enumerate(past_kv.layers):
        print(f"  Layer {i}: seq_len = {layer.keys.shape[2]}")



Step 0
  Layer 0: seq_len = 1
  Layer 1: seq_len = 1
  Layer 2: seq_len = 1
  Layer 3: seq_len = 1
  Layer 4: seq_len = 1
  Layer 5: seq_len = 1
  Layer 6: seq_len = 1
  Layer 7: seq_len = 1
  Layer 8: seq_len = 1
  Layer 9: seq_len = 1
  Layer 10: seq_len = 1
  Layer 11: seq_len = 1

Step 1
  Layer 0: seq_len = 2
  Layer 1: seq_len = 2
  Layer 2: seq_len = 2
  Layer 3: seq_len = 2
  Layer 4: seq_len = 2
  Layer 5: seq_len = 2
  Layer 6: seq_len = 2
  Layer 7: seq_len = 2
  Layer 8: seq_len = 2
  Layer 9: seq_len = 2
  Layer 10: seq_len = 2
  Layer 11: seq_len = 2

Step 2
  Layer 0: seq_len = 3
  Layer 1: seq_len = 3
  Layer 2: seq_len = 3
  Layer 3: seq_len = 3
  Layer 4: seq_len = 3
  Layer 5: seq_len = 3
  Layer 6: seq_len = 3
  Layer 7: seq_len = 3
  Layer 8: seq_len = 3
  Layer 9: seq_len = 3
  Layer 10: seq_len = 3
  Layer 11: seq_len = 3


**Define the Router**

In [ ]:
def kv_nonzero_elements(past_kv):
    total = 0

    for layer in past_kv.layers:

        for attr_name in dir(layer):
            attr = getattr(layer, attr_name)

            if hasattr(attr, "numel"):  # it's a tensor
                total += (attr != 0).sum().item()

    return total


In [ ]:
# -------------------------------------------------
# Router
# -------------------------------------------------
def router(layer_idx):
    if layer_idx <= 3:
        return "topk"
    elif layer_idx <= 6:
        return "head"
    elif layer_idx <= 8:
        return "layer_sparse"
    else:
        return "full"





**Expert Selection without importance**

**KV Memory Counter**

In [ ]:
def kv_elements(past_kv):
    total = 0
    for k, v in past_kv:
        total += k.numel() + v.numel()
    return total

**Routing Without Importance**

**Default KV Caching**

In [ ]:
inputs = tokenizer("The future of AI is", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
past_kv = None

def kv_elements(past_kv):
    total = 0
    for layer in past_kv.layers:
        total += layer.keys.numel()
        total += layer.values.numel()
    return total


for step in range(20):
    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)
    past_kv = out.past_key_values
    input_ids = torch.cat([input_ids, next_token], dim=-1)

    print(
        f"Step {step:02d} | "
        f"KV elements: {kv_elements(past_kv)}"
    )


Step 00 | KV elements: 18432
Step 01 | KV elements: 36864
Step 02 | KV elements: 55296
Step 03 | KV elements: 73728
Step 04 | KV elements: 92160
Step 05 | KV elements: 110592
Step 06 | KV elements: 129024
Step 07 | KV elements: 147456
Step 08 | KV elements: 165888
Step 09 | KV elements: 184320
Step 10 | KV elements: 202752
Step 11 | KV elements: 221184
Step 12 | KV elements: 239616
Step 13 | KV elements: 258048
Step 14 | KV elements: 276480
Step 15 | KV elements: 294912
Step 16 | KV elements: 313344
Step 17 | KV elements: 331776
Step 18 | KV elements: 350208
Step 19 | KV elements: 368640


**Gating Features Define**

In [ ]:
def token_importance(attn_weights):
    # use mean of top-5% instead of max
    last_attn = attn_weights[:, :, -1, :]
    topk_vals, _ = torch.topk(last_attn, k=max(1, last_attn.shape[-1] // 20))
    return topk_vals.mean().item()


import torch

def attention_entropy(attn_weights):
    # average over heads, use last query
    p = attn_weights[:, :, -1, :].mean(dim=1)
    entropy = -(p * torch.log(p + 1e-8)).sum(dim=-1)
    return entropy.mean().item()

def memory_pressure(past_kv, max_kv_elements):
    if past_kv is None:
        return 0.0
    return kv_elements(past_kv) / max_kv_elements



**Build the Gating Network**

In [ ]:
def gating_router(layer_idx, token_imp, attn_entropy, mem_pressure):

    # --------------------------------------
    # 1️⃣ Very High Pressure → aggressive
    # --------------------------------------
    if mem_pressure > 0.7:
        if layer_idx < 3:
            return "topk"
        elif layer_idx < 8:
            return "head"
        else:
            return "layer_sparse"

    # --------------------------------------
    # 2️⃣ High Pressure → strong pruning
    # --------------------------------------
    if mem_pressure > 0.5:
        if layer_idx < 4:
            return "topk"
        elif layer_idx < 10:
            return "head"
        else:
            return "layer_sparse"

    # --------------------------------------
    # 3️⃣ Medium Pressure → moderate pruning
    # --------------------------------------
    if mem_pressure > 0.2:
        if attn_entropy > 1.5:
            return "topk"
        else:
            return "head"

    # --------------------------------------
    # 4️⃣ Low Pressure → light pruning
    # --------------------------------------
    if token_imp < 0.5:
        return "topk"

    return "full"


In [ ]:
from transformers.cache_utils import DynamicCache
import torch

# assumes you already defined:
# token_importance, attention_entropy, memory_pressure
# gating_router(layer_idx, t_imp, ent, mem_p)
# experts = {...} mapping expert_name -> expert object
# and model/tokenizer/device exist

inputs = tokenizer("The future of AI is", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

analysis_cache = None   # for attentions ONLY (raw cache)
exec_cache = None       # for MoE-KV ONLY (modified cache)

MAX_KV_ELEMENTS = 600_000

for step in range(20):

    # =========================
    # PASS 1 — ANALYSIS (NO KV MODIFICATION)
    # =========================
    with torch.no_grad():
        analysis_out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=analysis_cache,
            use_cache=True,
            output_attentions=True
        )

    routing = []
    mem_p = memory_pressure(exec_cache, MAX_KV_ELEMENTS)  # uses exec_cache from previous step

    for layer_idx, attn in enumerate(analysis_out.attentions):
        t_imp = token_importance(attn)
        ent = attention_entropy(attn)

        expert_name = gating_router(layer_idx, t_imp, ent, mem_p)
        routing.append(expert_name)

        print(
            f"Step {step:02d} | Layer {layer_idx:02d} "
            f"| imp={t_imp:.2f} ent={ent:.2f} memP={mem_p:.3f} → {expert_name}"
        )

    # update analysis cache (RAW, UNMODIFIED)
    analysis_cache = analysis_out.past_key_values


    # =========================
    # PASS 2 — EXECUTION (MoE-KV MODIFICATION)
    # =========================
    with torch.no_grad():
        exec_out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=exec_cache,
            use_cache=True,
            output_attentions=False
        )

    next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)

    # Start from model-produced DynamicCache, then modify layers in-place
    new_exec_cache = exec_out.past_key_values  # DynamicCache

    for layer_idx, layer in enumerate(new_exec_cache.layers):
        expert = experts[routing[layer_idx]]

        # IMPORTANT: expert.update must accept (past_layer, new_layer, layer_idx)
        updated_layer = expert.update(
            exec_cache.layers[layer_idx] if exec_cache else None,
            layer,
            layer_idx
        )

        new_exec_cache.layers[layer_idx] = updated_layer

    exec_cache = new_exec_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)


TypeError: 'NoneType' object is not subscriptable

The Error is because GPT-2 internally assumes monotonic, unmodified KV growth when building the causal mask.

In [ ]:
def count_kv_elements(past_kv):
    if past_kv is None:
        return 0

    total = 0
    for layer in past_kv.layers:
        total += layer.keys.numel()
        total += layer.values.numel()

    return total

def estimate_full_kv(past_key_values):

    if past_key_values is None:
        return 0

    total = 0

    # Case: DynamicCache
    if hasattr(past_key_values, "layers"):

        for layer in past_key_values.layers:

            if layer is None:
                continue

            # Some models may partially initialize layers
            if not hasattr(layer, "keys") or not hasattr(layer, "values"):
                continue

            if layer.keys is None or layer.values is None:
                continue

            total += layer.keys.numel()
            total += layer.values.numel()

        return total

    # Fallback: tuple-style cache
    if isinstance(past_key_values, (list, tuple)):

        for layer in past_key_values:

            if layer is None:
                continue

            if not isinstance(layer, (list, tuple)):
                continue

            if len(layer) < 2:
                continue

            k, v = layer[0], layer[1]

            if k is None or v is None:
                continue

            total += k.numel() + v.numel()

        return total

    return 0






def estimate_topk_only(past_kv, K=16):
    total = 0

    for layer in past_kv:
        k = layer[0]
        v = layer[1]

        seq_len = k.shape[2]
        keep = min(K, seq_len)

        total += (
            k[:, :, -keep:, :].numel() +
            v[:, :, -keep:, :].numel()
        )

    return total


def estimate_head_only(past_kv, H_KEEP=2):
    total = 0

    for layer in past_kv:
        k = layer[0]   # first element is key
        v = layer[1]   # second element is value

        heads = k.shape[1]
        keep = min(H_KEEP, heads)

        total += (
            k[:, :keep, :, :].numel() +
            v[:, :keep, :, :].numel()
        )

    return total


def estimate_layer_sparse_only(past_kv, keep_last=4):
    total = 0
    total_layers = len(past_kv)

    for i, layer in enumerate(past_kv):

        if i >= total_layers - keep_last:
            k = layer[0]
            v = layer[1]
            total += k.numel() + v.numel()

    return total

def estimate_kv_after_routing(past_kv, routing, K_topk=4, H_keep=3, keep_last_layers=4):
    total = 0
    n_layers = len(past_kv)

    for i, (layer, route) in enumerate(zip(past_kv, routing)):
        k = layer[0]
        v = layer[1]

        b, h, s, d = k.shape  # (batch, heads, seq_len, head_dim)

        if route == "full":
            total += k.numel() + v.numel()

        elif route == "topk":
            keep = min(K_topk, s)
            total += k[:, :, -keep:, :].numel() + v[:, :, -keep:, :].numel()

        elif route == "head":
            keep_h = min(H_keep, h)
            total += k[:, :keep_h, :, :].numel() + v[:, :keep_h, :, :].numel()

        elif route == "layer_sparse":
            # keep KV only for last N layers
            if i >= (n_layers - keep_last_layers):
                total += k.numel() + v.numel()
            else:
                total += 0

        else:
            # if route string is unexpected, assume full (safe)
            total += k.numel() + v.numel()

    return total



In [ ]:
# =================================================
# GENERATION + ROUTED KV ESTIMATION (FIXED)
# =================================================

results = []

past_kv = None
input_ids = tokenizer("Hello", return_tensors="pt").input_ids.to(device)

for step in range(19):

    with torch.no_grad():
        out = model(
            input_ids=input_ids[:, -1:],
            past_key_values=past_kv,
            use_cache=True,
            output_attentions=True
        )

    past_kv = out.past_key_values

    # -------------------------
    # Compute Full KV
    # -------------------------
    full_kv_now = estimate_full_kv(past_kv)

    # Proper memory pressure normalization (grows from 0→1)
    max_full_possible = full_kv_now  # dynamic normalization
    mem_pressure = full_kv_now / max_full_possible if max_full_possible > 0 else 0

    # -------------------------
    # Dynamic Routing
    # -------------------------
    routing = []

    for layer_idx, attn in enumerate(out.attentions):

        t_imp = token_importance(attn)
        ent = attention_entropy(attn)

        expert_name = gating_router(
            layer_idx,
            t_imp,
            ent,
            mem_pressure
        )

        routing.append(expert_name)

    # -------------------------
    # KV Estimates
    # -------------------------
    topk_kv = estimate_topk_only(past_kv)
    head_kv = estimate_head_only(past_kv)
    sparse_kv = estimate_layer_sparse_only(past_kv)

    # 🔴 THIS WAS MISSING
    moe_kv_now = estimate_kv_after_routing(past_kv, routing)

    results.append({
        "Step": step,
        "Full": full_kv_now,
        "TopK": topk_kv,
        "HeadOnly": head_kv,
        "LayerSparse": sparse_kv,
        "MoE": moe_kv_now
    })

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)
    input_ids = torch.cat([input_ids, next_token], dim=1)


In [ ]:
print("\n" + "-"*110)
print(f"{'Step':<6} "
      f"{'Full':<12} "
      f"{'TopK':<12} "
      f"{'HeadOnly':<12} "
      f"{'LayerSparse':<14} "
      f"{'MoE':<12}")
print("-"*110)

for r in results:
    print(f"{r['Step']:<6} "
          f"{r['Full']:<12} "
          f"{r['TopK']:<12} "
          f"{r['HeadOnly']:<12} "
          f"{r['LayerSparse']:<14} "
          f"{r['MoE']:<12}")

print("-"*110)



--------------------------------------------------------------------------------------------------------------
Step   Full         TopK         HeadOnly     LayerSparse    MoE         
--------------------------------------------------------------------------------------------------------------
0      18432        18432        3072         6144           13824       
1      36864        36864        6144         12288          27648       
2      55296        55296        9216         18432          41472       
3      73728        73728        12288        24576          55296       
4      92160        92160        15360        30720          62976       
5      110592       110592       18432        36864          70656       
6      129024       129024       21504        43008          78336       
7      147456       147456       24576        49152          86016       
8      165888       165888       27648        55296          93696       
9      184320       184320       3072

In [ ]:
final = results[-1]
baseline = final["Full"]

def reduction(x, baseline):
    return (1 - x / baseline) * 100 if baseline > 0 else 0.0

print("\n=== FINAL STEP (Step 18) ===")
print(f"Full KV           : {final['Full']:,} (0.0%)")
print(f"Top-K only        : {final['TopK']:,} ({reduction(final['TopK'], baseline):.1f}%)")
print(f"Head-only         : {final['HeadOnly']:,} ({reduction(final['HeadOnly'], baseline):.1f}%)")
print(f"Layer-sparse only : {final['LayerSparse']:,} ({reduction(final['LayerSparse'], baseline):.1f}%)")
print(f"MoE-KV (ours)     : {final['MoE']:,} ({reduction(final['MoE'], baseline):.1f}%)")





=== FINAL STEP (Step 18) ===
Full KV           : 350,208 (0.0%)
Top-K only        : 294,912 (15.8%)
Head-only         : 58,368 (83.3%)
Layer-sparse only : 116,736 (66.7%)
MoE-KV (ours)     : 170,496 (51.3%)


In [ ]:
def route_layer_param(imp, ent, mem_p, mem_low, mem_high):

    # Low memory pressure → keep full
    if mem_p < mem_low:
        return "full"

    # Medium pressure → selective TopK
    if mem_p < mem_high:
        return "topk" if imp < 0.6 else "full"

    # High pressure → aggressive
    if ent < 1.5:
        return "head"

    return "layer_sparse"

def estimate_kv_after_routing(past_kv, routing, K=16, H_KEEP=[0,1,2]):
    if past_kv is None:
        return 0

    total = 0

    for i, layer in enumerate(past_kv.layers):
        route = routing[i]
        k = layer.keys
        v = layer.values

        if route == "full":
            total += k.numel() + v.numel()

        elif route == "topk":
            S = k.shape[2]
            keep = min(K, S)
            total += k[:, :, -keep:, :].numel()
            total += v[:, :, -keep:, :].numel()

        elif route == "head":
            total += k[:, H_KEEP, :, :].numel()
            total += v[:, H_KEEP, :, :].numel()

        elif route == "layer_sparse":
            continue

    return total

def compute_reduction(K, H_KEEP, steps=10):

    past_kv = None
    input_ids = tokenizer("Hello", return_tensors="pt").input_ids.to(device)

    total_full = 0
    total_moe = 0

    for step in range(steps):

        with torch.no_grad():
            out = model(
                input_ids=input_ids[:, -1:],
                past_key_values=past_kv,
                use_cache=True,
                output_attentions=True
            )

        past_kv = out.past_key_values

        # ---- Full KV ----
        full_kv = estimate_full_kv(past_kv)

        # ---- Routed KV (using K and H_KEEP) ----
        routing = []

        for layer_idx in range(len(past_kv)):

            # simple hybrid policy for tuning
            if layer_idx <= 3:
                routing.append("topk")
            elif layer_idx <= 6:
                routing.append("head")
            else:
                routing.append("full")

        moe_kv = estimate_kv_after_routing(
            past_kv,
            routing,
            K_topk=K,
            H_keep=len(H_KEEP)
        )

        total_full += full_kv
        total_moe += moe_kv

        next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)

    reduction = 100 * (1 - total_moe / total_full)

    return reduction, total_full, total_moe




In [ ]:
# Block 1: dataset + environment sanity check
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
print("Dataset size:", len(ds))

# Show a couple examples (to confirm it loaded correctly)
for i in range(2):
    txt = ds[i]["text"]
    print(f"\nExample {i} length:", len(txt))
    print("Example preview:", repr(txt[:120]))


Device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset size: 3760

Example 0 length: 0
Example preview: ''

Example 1 length: 23
Example preview: ' = Homarus gammarus = \n'


In [ ]:
# Block 2: Baseline GPT-2 Perplexity (small subset)

import math
from tqdm import tqdm

model.eval()

total_loss = 0
total_tokens = 0

subset = ds.select(range(300))

for example in tqdm(subset):

    text = example["text"].strip()
    if len(text) < 5:   # skip empty/short lines
        continue

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    if inputs["input_ids"].shape[1] < 2:
        continue

    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"],
            labels=inputs["input_ids"]
        )

    loss = outputs.loss
    n_tokens = inputs["input_ids"].numel()

    total_loss += loss.item() * n_tokens
    total_tokens += n_tokens

baseline_ppl = math.exp(total_loss / total_tokens)

print("\nBaseline GPT-2 Perplexity (subset):", baseline_ppl)


100%|██████████| 300/300 [02:39<00:00,  1.88it/s]


Baseline GPT-2 Perplexity (subset): 40.220014362629136


In [ ]:
# ==========================================================
# BLOCK 3 — MoE Perplexity (Single Sample, Fully Corrected)
# ==========================================================

import torch
import math

model.eval()

# ---- Pick one validation sample ----
example = ds[10]["text"].strip()
print("Sample length:", len(example))

if len(example) < 10:
    print("Sample too short — choose another index.")
else:

    inputs = tokenizer(
        example,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    input_ids = inputs["input_ids"]

    past_kv = None
    log_likelihood = 0.0
    token_count = 0

    for t in range(input_ids.shape[1] - 1):

        current_input = input_ids[:, t:t+1]

        with torch.no_grad():
            out = model(
                input_ids=current_input,
                past_key_values=past_kv,
                use_cache=True,
                output_attentions=True
            )

        logits = out.logits[:, -1, :]
        past_kv = out.past_key_values

        # ---- ROUTING SAFELY ----
        routing = []

        # If attentions not returned, default to full
        if out.attentions is None:
            routing = ["full"] * len(past_kv.layers)
        else:
            mem_p = memory_pressure(past_kv, MAX_KV_ELEMENTS)

            for attn in out.attentions:

                if attn is None:
                    routing.append("full")
                    continue

                imp = token_importance(attn)
                ent = attention_entropy(attn)

                expert = route_layer_param(
                    imp,
                    ent,
                    mem_p,
                    mem_low=0.2,
                    mem_high=0.5
                )

                routing.append(expert)

        # ---- LOG LIKELIHOOD ----
        next_token = input_ids[:, t+1]
        log_probs = torch.log_softmax(logits, dim=-1)

        log_likelihood += log_probs[0, next_token].item()
        token_count += 1

    moe_ppl = math.exp(-log_likelihood / token_count)

    print("MoE Perplexity (single sample):", moe_ppl)


Sample length: 308
MoE Perplexity (single sample): 32.30075633812753


In [ ]:
# ==========================================================
# BLOCK 4 — MoE Perplexity (300 Validation Samples)
# ==========================================================

import math
from tqdm import tqdm

model.eval()

total_log_likelihood = 0.0
total_tokens = 0

subset = ds.select(range(300))

for example in tqdm(subset):

    text = example["text"].strip()
    if len(text) < 5:
        continue

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    input_ids = inputs["input_ids"]

    past_kv = None

    for t in range(input_ids.shape[1] - 1):

        current_input = input_ids[:, t:t+1]

        with torch.no_grad():
            out = model(
                input_ids=current_input,
                past_key_values=past_kv,
                use_cache=True,
                output_attentions=True
            )

        logits = out.logits[:, -1, :]
        past_kv = out.past_key_values

        # ---- ROUTING ----
        if out.attentions is None:
            routing = ["full"] * len(past_kv.layers)
        else:
            mem_p = memory_pressure(past_kv, MAX_KV_ELEMENTS)
            routing = []

            for attn in out.attentions:
                if attn is None:
                    routing.append("full")
                    continue

                imp = token_importance(attn)
                ent = attention_entropy(attn)

                expert = route_layer_param(
                    imp,
                    ent,
                    mem_p,
                    mem_low=0.2,
                    mem_high=0.5
                )

                routing.append(expert)

        next_token = input_ids[:, t+1]
        log_probs = torch.log_softmax(logits, dim=-1)

        total_log_likelihood += log_probs[0, next_token].item()
        total_tokens += 1

moe_ppl = math.exp(-total_log_likelihood / total_tokens)

print("\nMoE Perplexity (300 samples):", moe_ppl)


100%|██████████| 300/300 [20:08<00:00,  4.03s/it]


MoE Perplexity (300 samples): 40.283122525676816


In [ ]:
# ==========================================================
# BLOCK 5 — TRUE MoE EXECUTION (Single Sample Test)
# ==========================================================

import math
from transformers.cache_utils import DynamicCache

model.eval()

example = ds[10]["text"].strip()
print("Sample length:", len(example))

inputs = tokenizer(
    example,
    return_tensors="pt",
    truncation=True,
    max_length=256
).to(device)

input_ids = inputs["input_ids"]

analysis_cache = None
exec_cache = None

log_likelihood = 0.0
token_count = 0

for t in range(input_ids.shape[1] - 1):

    current_input = input_ids[:, t:t+1]

    # ========================
    # PASS 1 — ANALYSIS
    # ========================
    with torch.no_grad():
        analysis_out = model(
            input_ids=current_input,
            past_key_values=analysis_cache,
            use_cache=True,
            output_attentions=True
        )

    analysis_cache = analysis_out.past_key_values

    mem_p = memory_pressure(exec_cache, MAX_KV_ELEMENTS)
    routing = []

    for attn in analysis_out.attentions:
        if attn is None:
            routing.append("full")
            continue

        imp = token_importance(attn)
        ent = attention_entropy(attn)

        routing.append(
            route_layer_param(imp, ent, mem_p, 0.2, 0.5)
        )

    # ========================
    # PASS 2 — EXECUTION (MoE KV)
    # ========================
    with torch.no_grad():
        exec_out = model(
            input_ids=current_input,
            past_key_values=exec_cache,
            use_cache=True,
            output_attentions=False
        )

    logits = exec_out.logits[:, -1, :]

    # Build new modified cache
    new_cache = DynamicCache()

    for i, layer in enumerate(exec_out.past_key_values.layers):

        k = layer.keys
        v = layer.values

        route = routing[i]

        if route == "topk":
            K = 16
            S = k.shape[2]
            keep = min(K, S)
            k = k[:, :, -keep:, :]
            v = v[:, :, -keep:, :]

        elif route == "head":
            keep_heads = [0,1,2]
            k = k[:, keep_heads, :, :]
            v = v[:, keep_heads, :, :]

        elif route == "layer_sparse":
            k = torch.zeros_like(k)
            v = torch.zeros_like(v)

        # full = unchanged

        new_cache.update(k, v, i)

    exec_cache = new_cache

    # ========================
    # LOG LIKELIHOOD
    # ========================
    next_token = input_ids[:, t+1]
    log_probs = torch.log_softmax(logits, dim=-1)

    log_likelihood += log_probs[0, next_token].item()
    token_count += 1

true_moe_ppl = math.exp(-log_likelihood / token_count)

print("TRUE MoE Perplexity (single sample):", true_moe_ppl)


Sample length: 308
TRUE MoE Perplexity (single sample): 32.30075633812753


In [ ]:
def count_nonzero_kv(cache):
    if cache is None:
        return 0
    total = 0
    for layer in cache.layers:
        total += torch.count_nonzero(layer.keys).item()
        total += torch.count_nonzero(layer.values).item()
    return total

def apply_routing_to_cache_inplace(cache, routing, K=16, H_KEEP=(0,1,2), keep_last_layers=4):
    """
    In-place: zero out parts of KV to simulate routed caching.
    Shapes stay identical so the model won't crash.
    """
    if cache is None:
        return cache

    n_layers = len(cache.layers)

    for i, layer in enumerate(cache.layers):
        k = layer.keys
        v = layer.values
        route = routing[i] if i < len(routing) else "full"

        if route == "full":
            continue

        elif route == "topk":
            # keep only last K tokens, zero the rest (seq dim = 2)
            S = k.shape[2]
            keep = min(K, S)
            if S > keep:
                k[:, :, :S-keep, :] = 0
                v[:, :, :S-keep, :] = 0

        elif route == "head":
            # keep only specific heads, zero the others (head dim = 1)
            heads = k.shape[1]
            keep_set = set(int(h) for h in H_KEEP if int(h) < heads)
            if len(keep_set) == 0:
                # if nothing valid, fallback to full
                continue
            mask = torch.ones(heads, dtype=torch.bool, device=k.device)
            for h in keep_set:
                mask[h] = False
            k[:, mask, :, :] = 0
            v[:, mask, :, :] = 0

        elif route == "layer_sparse":
            # keep only the last N layers
            if i < (n_layers - keep_last_layers):
                k.zero_()
                v.zero_()

        else:
            # unknown route -> safe fallback
            continue

    return cache


def build_routing(analysis_out, num_layers, mem_p):
    routing = []
    for layer_idx in range(num_layers):

        # If attentions missing, force router to choose a non-full path sometimes
        if analysis_out.attentions is None or layer_idx >= len(analysis_out.attentions) or analysis_out.attentions[layer_idx] is None:
            imp = 0.0     # low importance -> tends to topk
            ent = 10.0    # high entropy -> tends to topk in your logic
        else:
            attn = analysis_out.attentions[layer_idx]
            imp = token_importance(attn)
            ent = attention_entropy(attn)

        routing.append(gating_router(layer_idx, imp, ent, mem_p))

    return routing



In [ ]:
from collections import Counter
prompts = [
    "User: Explain climate change risks.\nAssistant:",
    "User: What are the dangers of AI in society?\nAssistant:",
    "User: How can governments regulate AI safely?\nAssistant:",
    "User: Describe ethical concerns in artificial intelligence.\nAssistant:"
]


def run_eval(prompts, steps=120, K=16, H_KEEP=(0,1,2), keep_last_layers=4, use_moe=False):
    rows = []

    for idx, prompt in enumerate(prompts, 1):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs["input_ids"]

        analysis_cache = None
        exec_cache = None

        # track “effective” memory as non-zero KV (this is what pruning changes)
        total_nonzero = 0
        routing_counter = Counter()

        MAX_KV = 0

        for t in range(steps):
            current_input = input_ids[:, -1:]

            # PASS 1: analysis (try attentions, but we can survive without)
            with torch.no_grad():
                analysis_out = model(
                    input_ids=current_input,
                    past_key_values=analysis_cache,
                    use_cache=True,
                    output_attentions=True,
                    return_dict=True
                )
            analysis_cache = analysis_out.past_key_values

            # pressure based on exec_cache growth
            current_full = estimate_full_kv(exec_cache)
            MAX_KV = max(MAX_KV, current_full)
            mem_p = (current_full / MAX_KV) if MAX_KV > 0 else 0.0

            num_layers = len(analysis_cache.layers) if hasattr(analysis_cache, "layers") else len(analysis_cache)
            routing = build_routing(analysis_out, num_layers, mem_p)

            # PASS 2: execution
            with torch.no_grad():
                exec_out = model(
                    input_ids=current_input,
                    past_key_values=exec_cache,
                    use_cache=True,
                    return_dict=True
                )
            exec_cache = exec_out.past_key_values

            # APPLY MoE pruning to the cache (this is the key difference)
            if use_moe and hasattr(exec_cache, "layers"):
                exec_cache = apply_routing_to_cache_inplace(
                    exec_cache, routing, K=K, H_KEEP=H_KEEP, keep_last_layers=keep_last_layers
                )
                routing_counter.update(routing)

            # next token
            next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=1)

            total_nonzero += count_nonzero_kv(exec_cache) if hasattr(exec_cache, "layers") else 0

        # final conversation text
        text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

        rows.append({
            "Conversation": idx,
            "Text": text,
            "Full_KV": estimate_full_kv(exec_cache),
            "NonZero_KV_sum": total_nonzero,
            "RoutingDist": dict(routing_counter) if use_moe else {}
        })

    return rows


# -------- run baseline vs MoE --------
baseline = run_eval(prompts, steps=120, use_moe=False)
moe = run_eval(prompts, steps=120, K=16, H_KEEP=(0,1,2), keep_last_layers=4, use_moe=True)

print("\n=== BASELINE vs MoE (NonZero KV) ===\n")
print(f"{'Conv':<6}{'BaselineNZ':<15}{'MoE_NZ':<15}{'Reduction%':<12}")
print("-"*52)

for b, m in zip(baseline, moe):
    base_nz = b["NonZero_KV_sum"]
    moe_nz = m["NonZero_KV_sum"]
    red = (1 - moe_nz / base_nz) * 100 if base_nz > 0 else 0
    print(f"{b['Conversation']:<6}{base_nz:<15}{moe_nz:<15}{red:>10.2f}")

print("-"*52)

# -------- print conversations + routing --------
for m in moe:
    print("\n" + "="*90)
    print(f"Conversation {m['Conversation']} (MoE)")
    print("="*90)
    print(m["Text"])
    print("\nRouting Distribution:", m["RoutingDist"])



=== BASELINE vs MoE (NonZero KV) ===

Conv  BaselineNZ     MoE_NZ         Reduction%  
----------------------------------------------------
1     133816320      66820608            50.07
2     133816320      66820608            50.07
3     133816320      66820608            50.07
4     133816320      66820608            50.07
----------------------------------------------------

Conversation 1 (MoE)
User: Explain climate change risks.
Assistant:

"I'm not sure if I'm going to be able to say that I'm not only because I'm not going to be able to say that I'm not going to be able to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I'm not going to say that I

Routing Distribution: {'topk': 488, 'head': 476, 'full': 476}

Conversation 2 (MoE)
User: What are the dangers of 